# Support Vector Machines (SVMs)

**Dataset:** Iris (sklearn built-in)  
**Task:** Multiclass classification with multiple kernel types

---

## Overview

A **Support Vector Machine** finds the hyperplane that **maximizes the margin** between classes. The margin is the distance between the hyperplane and the nearest training points (the **support vectors**).

### Hard-Margin SVM (linearly separable case)

For a binary classification problem with labels $y_i \in \{-1, +1\}$:

$$\underset{\mathbf{w}, b}{\text{minimize}} \quad \frac{1}{2} \|\mathbf{w}\|^2 \quad \text{subject to} \quad y_i(\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 \;\; \forall i$$

The margin is $\frac{2}{\|\mathbf{w}\|}$.

### Soft-Margin SVM

For noisy/overlapping data, we allow slack variables $\xi_i \geq 0$:

$$\underset{\mathbf{w}, b, \xi}{\text{minimize}} \quad \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_i \xi_i$$

The hyperparameter $C$ controls the **bias-variance tradeoff**: small $C$ = wider margin (more regularized), large $C$ = narrower margin (fits training data more closely).

### The Kernel Trick

SVMs can model nonlinear boundaries by implicitly mapping data to a higher-dimensional space using a **kernel function** $K(\mathbf{x}_i, \mathbf{x}_j)$:

| Kernel | Formula | Use Case |
|--------|---------|----------|
| Linear | $\mathbf{x}_i^\top \mathbf{x}_j$ | Linearly separable data |
| Polynomial | $(\mathbf{x}_i^\top \mathbf{x}_j + r)^d$ | Polynomial boundaries |
| RBF (Gaussian) | $\exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$ | Most flexible, default choice |

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 2. Load Data

In [ ]:
data = load_iris()
X, y = data.data, data.target
feature_names = data.feature_names
target_names  = data.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape[0]} | Test: {X_test_s.shape[0]}")

## 3. Compare Kernel Types

In [ ]:
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
kernel_results = {}

for k in kernels:
    svm = SVC(kernel=k, C=1.0, random_state=42)
    svm.fit(X_train_s, y_train)
    train_acc = svm.score(X_train_s, y_train)
    test_acc  = svm.score(X_test_s,  y_test)
    kernel_results[k] = (train_acc, test_acc)
    print(f"Kernel: {k:8s} | Train: {train_acc:.4f} | Test: {test_acc:.4f} | Support Vectors: {svm.n_support_}")

## 4. Decision Boundaries for Different Kernels (2D)

We use only the 2 petal features for visualization.

In [ ]:
X2 = X[:, 2:4]  # petal features
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y
)
scaler2 = StandardScaler()
X2_train_s = scaler2.fit_transform(X2_train)
X2_test_s  = scaler2.transform(X2_test)

x_min, x_max = X2_train_s[:, 0].min() - 0.5, X2_train_s[:, 0].max() + 0.5
y_min, y_max = X2_train_s[:, 1].min() - 0.5, X2_train_s[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

colors = ['#e74c3c', '#2ecc71', '#3498db']
kernels_plot = ['linear', 'poly', 'rbf']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, k in zip(axes, kernels_plot):
    svm = SVC(kernel=k, C=1.0, random_state=42)
    svm.fit(X2_train_s, y2_train)
    acc = svm.score(X2_test_s, y2_test)

    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlBu')

    for cls, color, label in zip([0, 1, 2], colors, target_names):
        mask = y2_train == cls
        ax.scatter(X2_train_s[mask, 0], X2_train_s[mask, 1],
                   c=color, label=label, edgecolors='k', s=50, alpha=0.8)

    # Mark support vectors
    sv = svm.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=150, facecolors='none',
               edgecolors='black', linewidths=1.5, label='Support vectors')

    ax.set_title(f'Kernel: {k} (acc={acc:.2f})')
    ax.set_xlabel(feature_names[2])
    if ax == axes[0]:
        ax.set_ylabel(feature_names[3])
    ax.legend(fontsize=8)

plt.suptitle('SVM Decision Boundaries — Petal Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Effect of the Regularization Parameter C

$C$ controls the tradeoff between maximizing the margin and minimizing misclassification on training data. A small $C$ allows more violations (wider margin, more regularized); a large $C$ enforces stricter classification (smaller margin, less regularized).

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
train_accs, test_accs = [], []

for C in C_values:
    svm = SVC(kernel='rbf', C=C, random_state=42)
    svm.fit(X_train_s, y_train)
    train_accs.append(svm.score(X_train_s, y_train))
    test_accs.append(svm.score(X_test_s, y_test))

plt.figure(figsize=(8, 5))
plt.semilogx(C_values, train_accs, 'o-', color='steelblue', label='Train accuracy')
plt.semilogx(C_values, test_accs,  's--', color='tomato',    label='Test accuracy')
plt.xlabel('C (log scale)')
plt.ylabel('Accuracy')
plt.title('SVM (RBF Kernel): Effect of C')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.1, 1],
}

grid_search = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_s, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")
print(f"Test accuracy:    {grid_search.score(X_test_s, y_test):.4f}")

## 7. Final Evaluation

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)
ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
ax.set_xticklabels(target_names, rotation=20)
ax.set_yticklabels(target_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Best SVM — Confusion Matrix')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=13)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=target_names))

## 8. Summary

### Key Takeaways

- SVMs find the **maximum-margin hyperplane** — maximizing the gap between classes makes the classifier more robust to noise.
- **Support vectors** are the training points closest to the boundary; only they determine the decision surface.
- The **kernel trick** allows SVMs to learn nonlinear boundaries without explicitly computing transformations.
- **RBF kernel** is the most flexible and commonly used default.
- **Feature scaling** is critical — SVM distances are sensitive to feature magnitude.
- Hyperparameters $C$ (regularization) and $\gamma$ (RBF bandwidth) are best tuned via cross-validated grid search.